# Multimodal LLMs and Deep Learning for Nuclei Image Analysis

Four tasks on a synthetic fluorescence microscopy dataset of stained cell nuclei:

1. Data preparation and multimodal description, grayscale, resize, EDA, then describe an image with llama3.2-vision using a naive prompt and an engineered structured prompt.
2. Classical features and LLM interpretation, Otsu, morphology, regionprops_table, then narrate the numbers with a text-only model that never sees the image.
3. U-Net segmentation, train a small PyTorch U-Net, evaluate mean Dice and IoU on validation.
4. Hybrid pipeline, U-Net mask, feature table, LLM record and narrative for every test image, aggregated to a CSV.

Run this on Colab with a GPU runtime (Runtime, Change runtime type, T4 GPU).

## Setup

Installs packages, starts Ollama inside the Colab VM and pulls the models. The vision model is about 8 GB, so this cell takes several minutes the first time.

In [ ]:
import os, sys, subprocess, time, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("running in Colab:", IN_COLAB)

if IN_COLAB:
    # GPU check
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "no GPU")

    !pip install -q ollama scikit-image
    # the Ollama installer needs zstd, which the Colab image does not ship
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh

    # confirm the binary landed before trying to start it
    ollama_bin = subprocess.run(["which", "ollama"], capture_output=True,
                                text=True).stdout.strip()
    print("ollama binary:", ollama_bin or "NOT FOUND")

    if ollama_bin:
        server = subprocess.Popen([ollama_bin, "serve"],
                                  stdout=subprocess.DEVNULL,
                                  stderr=subprocess.DEVNULL)
        for _ in range(30):
            try:
                urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
                print("ollama is up"); break
            except Exception:
                time.sleep(2)
        else:
            print("ollama did not start; check the install output above")

        # llama3.2-vision uses the mllama architecture, which recent
        # Ollama builds refuse to load. Qwen2.5-VL is the sanctioned
        # alternative and is pulled as the working vision model.
        !ollama pull llama3.2-vision
        !ollama pull qwen2.5vl:7b
        !ollama pull llama3.2
        !ollama pull qwen2.5:3b
        !ollama list

### Project code and dataset

Upload nuclei-project-colab.zip when prompted. It contains src/, tests/, the notebook and the dataset archive. Alternatively set REPO to a git URL and it will clone instead.

In [ ]:
REPO = ""   # optional: "https://github.com/USER/REPO.git"

if IN_COLAB:
    if REPO:
        subprocess.run(["git", "clone", "-q", REPO, "project"], check=True)
        os.chdir("project")
    elif not Path("src").exists():
        # upload nuclei-project-colab.zip (contains src/, tests/, the dataset)
        from google.colab import files
        print("Upload nuclei-project-colab.zip")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                subprocess.run(["unzip", "-q", "-o", name], check=True)
    if not Path("data/nuclei_dataset").exists():
        Path("data").mkdir(exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", "nuclei_dataset.zip", "-d", "data"],
                       check=True)

sys.path.insert(0, "src")
import warnings; warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

import data_prep, classical, metrics, eda, visualise, llm, prompts, pipeline
print("modules loaded")

# Task 1: data preparation and multimodal description

## Loading, grayscale conversion and resizing

The images are DAPI-like blue-stained nuclei on a dark field. Standard luminance grayscale weights blue at only 0.114, which dims exactly the signal we care about, so the loader keeps the blue channel instead. Everything is resized to 256 by 256 and scaled to the range 0 to 1.

In [ ]:
md_table = data_prep.metadata()
print("images:", len(md_table))
display(md_table.groupby(["split", "density"]).size().unstack(fill_value=0))

for split in data_prep.SPLITS:
    ids, imgs, masks = data_prep.load_split(split)
    print(f"{split}: {len(ids)} images, shape {imgs.shape[1:]}, "
          f"intensity range [{imgs.min():.2f}, {imgs.max():.2f}], "
          f"nucleus pixels {masks.mean():.1%}")

## Exploratory analysis

In [ ]:
summary = eda.run()
for f in ["eda_samples.png", "eda_intensity.png",
          "eda_grayscale.png", "eda_corrupted.png"]:
    display(ShowImage(filename=f"outputs/figures/{f}"))

## Multimodal description: naive against structured prompt

Both prompts go to llama3.2-vision with the same image. The naive prompt has no role, no format and no permission to be unsure. The structured prompt anchors the model as descriptive rather than diagnostic, fixes the JSON schema, and allows "uncertain" for any field it cannot determine.

In [ ]:
from skimage import io as skio

# available_vision_model tries each candidate with a one-token request,
# so a model that downloads but fails to load is skipped automatically
VISION = llm.available_vision_model()
print("vision model in use:", VISION)

# save a representative image as PNG for the model to read
sample_id = "train_001"
rgb = skio.imread(data_prep.DATA / "train" / "images" / f"{sample_id}.png")
Path("outputs/samples").mkdir(parents=True, exist_ok=True)
sample_path = f"outputs/samples/{sample_id}.png"
skio.imsave(sample_path, rgb)
display(ShowImage(filename=sample_path))

In [ ]:
print("=" * 70, "\nNAIVE PROMPT\n", "=" * 70)
print(prompts.VISION_NAIVE, "\n")
naive = llm.generate(prompts.VISION_NAIVE, model=VISION, images=[sample_path])
print(naive)

In [ ]:
print("=" * 70, "\nSTRUCTURED PROMPT\n", "=" * 70)
print(prompts.VISION_STRUCTURED, "\n")
structured = llm.generate(prompts.VISION_STRUCTURED, model=VISION,
                          images=[sample_path])
print(structured)
print("\nparsed:", llm.extract_json(structured))

## Repeated runs are not identical

Same image, same prompt, temperature 0.8, cache disabled. The outputs differ, which is why any single description should be treated as one sample rather than the answer.

In [ ]:
for i in range(3):
    out = llm.generate(prompts.VISION_STRUCTURED, model=VISION,
                       images=[sample_path], temperature=0.8, use_cache=False)
    record = llm.extract_json(out)
    print(f"run {i+1}: {record}\n" if record else f"run {i+1}: unparsed\n{out[:200]}\n")

## Hallucination and the uncertainty escape hatch

The image cannot possibly show a patient age or sex. The first prompt invites the model to answer anyway; the second gives it an explicit way out.

In [ ]:
print("--- inviting a hallucination ---")
print(llm.generate(prompts.VISION_HALLUCINATION, model=VISION,
                   images=[sample_path]), "\n")

print("--- with an escape hatch ---")
print(llm.generate(prompts.VISION_HALLUCINATION_SAFE, model=VISION,
                   images=[sample_path]))

# Task 2: classical features and LLM interpretation

## Otsu, morphological cleanup and labelling

Opening removes speckle, closing seals gaps in nucleus edges, then small objects and pinholes go. Connected components would treat any touching group as one object, so a distance-transform watershed splits them. The separation parameter was tuned on the training split only.

In [ ]:
visualise.segmentation_steps()
display(ShowImage(filename="outputs/figures/classical_steps.png"))

## Per-object feature table

In [ ]:
demo_id = data_prep.image_ids("val")[0]
demo_img = data_prep.load_image("val", demo_id)
demo_summary, demo_table, demo_labels = classical.analyse(demo_img)

print(f"{demo_id}: {len(demo_table)} objects detected")
display(demo_table.head(8).round(3))
print("\nimage-level summary passed to the LLM:")
for k, v in demo_summary.items():
    print(f"  {k}: {v}")

## How well does the classical method do?

Two different questions: Dice and IoU ask whether each pixel is classified correctly, count error asks whether the right number of nuclei was found. A method can be excellent at the first and poor at the second.

In [ ]:
rows = []
md_idx = data_prep.metadata().set_index("image_id")
for split in ["val", "test"]:
    for iid in data_prep.image_ids(split):
        img = data_prep.load_image(split, iid)
        truth = data_prep.load_mask(split, iid)
        binary, _ = classical.otsu_mask(img)
        s, _, _ = classical.analyse(img)
        rows.append({"split": split, "density": md_idx.loc[iid, "density"],
                     **metrics.pixel_scores(binary, truth),
                     **metrics.count_scores(s["n_objects"],
                                            md_idx.loc[iid, "n_objects"])})

print("classical baseline, held-out images")
display(metrics.summarise_scores(rows).to_frame("mean").T)
display(metrics.summarise_scores(rows, by="density")
        [["dice", "iou", "n_true", "n_pred", "abs_count_error"]])

### Classical segmentation against ground truth

The same side-by-side view used later for the U-Net, so the two segmentation routes can be compared on identical images.

In [ ]:
val_ids_demo = data_prep.image_ids("val")[:3]
otsu_masks = [classical.otsu_mask(data_prep.load_image("val", i))[0]
              for i in val_ids_demo]

visualise.prediction_grid("val", val_ids_demo, otsu_masks,
                          "classical_predictions.png",
                          "Classical Otsu segmentation on validation images",
                          n_show=3)
display(ShowImage(filename="outputs/figures/classical_predictions.png"))

## Numbers-first description

The text model receives the measurements only. It has never seen the image, and the prompt says so. A second model then audits the paragraph against the same numbers.

In [ ]:
paragraph, record, raw = pipeline.narrate(demo_summary)
print("MEASUREMENTS GIVEN TO THE MODEL")
print(pipeline.measurement_block(demo_summary))
print("\nPARAGRAPH\n", paragraph)
print("\nJSON RECORD\n", record)

scores = pipeline.audit(demo_summary, paragraph)
print("\nAUDIT by", llm.AUDIT_MODEL, "\n", scores)

## Comparing the two routes

Task 1 looked at the image. Task 2 measured it and described the measurements. The first is fluent but unverifiable; the second is auditable, because every claim traces back to a number in the feature table.

In [ ]:
print("VISION MODEL, looking at the image:")
print(llm.extract_json(structured), "\n")
print("NUMBERS-FIRST, never saw the image:")
print(record, "\n")
print(f"ground truth for {demo_id}: n_objects = {md_idx.loc[demo_id, 'n_objects']}, "
      f"density = {md_idx.loc[demo_id, 'density']}")

# Task 3: U-Net segmentation

A small U-Net, base 16 channels and 3 downsampling steps, roughly 0.5 M parameters. The training set is 80 images, so a full-size U-Net would have far more capacity than the data can constrain.

The loss is Dice plus binary cross entropy. About 92 per cent of pixels are background, so BCE alone rewards predicting background everywhere; the Dice term penalises exactly that.

In [ ]:
import unet

train_ids, train_imgs, train_masks = data_prep.load_split("train")
val_ids, val_imgs, val_masks = data_prep.load_split("val")
print(f"train {train_imgs.shape}, val {val_imgs.shape}")

train_loader, val_loader = unet.make_loaders(train_imgs, train_masks,
                                             val_imgs, val_masks,
                                             batch_size=8, augment=True)
model, history = unet.train_unet(train_loader, val_loader, epochs=30)

In [ ]:
visualise.training_curves(history)
display(ShowImage(filename="outputs/figures/unet_training.png"))
display(history.tail(5).round(4))

## Loss ablation

The same architecture, seed, optimiser, augmentation and epoch budget under three losses, so any difference is attributable to the loss alone. predicted_area_ratio is the total predicted nucleus area divided by the true nucleus area: a value well below 1 means the model is under-segmenting, which is the specific failure BCE alone is expected to allow.

In [ ]:
ablation, ablation_history = unet.loss_ablation(
    train_imgs, train_masks, val_imgs, val_masks, epochs=30)
display(ablation.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for name, g in ablation_history.groupby("loss_name"):
    axes[0].plot(g.epoch, g.val_dice, label=name)
    axes[1].plot(g.epoch, g.train_loss, label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("validation Dice")
axes[0].set_title("Validation Dice by loss"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("training loss")
axes[1].set_title("Training loss by loss function (scales differ)")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig("outputs/figures/loss_ablation.png", dpi=150)
plt.show()

ablation.to_csv("outputs/tables/loss_ablation.csv", index=False)
ablation_history.to_csv("outputs/tables/loss_ablation_history.csv", index=False)

## Validation Dice and IoU

In [ ]:
val_pred = unet.predict_masks(model, val_imgs)

unet_rows, otsu_rows = [], []
for i, iid in enumerate(val_ids):
    truth = val_masks[i]
    unet_rows.append({"image_id": iid, "density": md_idx.loc[iid, "density"],
                      **metrics.pixel_scores(val_pred[i], truth)})
    otsu_rows.append({"image_id": iid, "density": md_idx.loc[iid, "density"],
                      **metrics.pixel_scores(
                          classical.otsu_mask(val_imgs[i])[0], truth)})

comparison = pd.DataFrame({
    "U-Net": metrics.summarise_scores(unet_rows),
    "classical Otsu": metrics.summarise_scores(otsu_rows)}).T
print("validation split, pixel metrics")
display(comparison)

print("\nU-Net by density regime")
display(metrics.summarise_scores(unet_rows, by="density"))

## Input, ground truth and prediction side by side

In [ ]:
visualise.prediction_grid("val", val_ids[:4], val_pred[:4],
                          "unet_predictions.png",
                          "U-Net segmentation on validation images", n_show=4)
display(ShowImage(filename="outputs/figures/unet_predictions.png"))

# Task 4: hybrid pipeline on the unseen test images

For every test image: U-Net mask, then watershed instance labels, then regionprops features, then an LLM structured record and a one-paragraph narrative. The records are aggregated into a DataFrame and saved as CSV.

In [ ]:
test_ids, test_imgs, test_masks = data_prep.load_split("test")
test_pred = unet.predict_masks(model, test_imgs)

test_pixel = [{"image_id": iid, "density": md_idx.loc[iid, "density"],
               **metrics.pixel_scores(test_pred[i], test_masks[i])}
              for i, iid in enumerate(test_ids)]
print("U-Net on the unseen test split")
display(metrics.summarise_scores(test_pixel).to_frame("mean").T)

In [ ]:
records, objects, narratives = pipeline.run_test_set(
    masks=test_pred, split="test", narrate_records=True, do_audit=True)

display(records)
print("\nsaved to outputs/tables/test_records.csv")

In [ ]:
example = test_ids[0]
print(f"narrative for {example}:\n")
print(narratives[example])
print("\nfeature table head for this image:")
display(objects[objects.image_id == example].head(5).round(3))

## How accurate are the LLM records?

The density class is a deterministic function of the count, so it is right exactly when the count is right. Comparing against the ground truth regime shows where the pipeline loses nuclei.

In [ ]:
check = records[["image_id", "n_objects", "true_n_objects", "count_error",
                 "density_truth"]].copy()
if "llm_density_class" in records:
    check["llm_density_class"] = records["llm_density_class"]
display(check)

print(f"count MAE: {records.count_error.abs().mean():.2f}")
print(f"count bias: {records.count_error.mean():+.2f}")

visualise.count_scatter(records)
visualise.feature_distributions(objects)
display(ShowImage(filename="outputs/figures/count_accuracy.png"))
display(ShowImage(filename="outputs/figures/feature_distributions.png"))

## Robustness on the corrupted images

The same scene blurred and at low contrast. This is the failure case the quality_flag field exists for.

In [ ]:
for iid in data_prep.corrupted_ids():
    img = data_prep.load_image("test", iid, corrupted=True)
    s, _, _ = classical.analyse(img)
    base = iid.split("_")[0] + "_" + iid.split("_")[1]
    print(f"{iid:26} n_objects {s['n_objects']:3d}  "
          f"mean_intensity {s['mean_intensity']:.3f}  "
          f"(clean truth {md_idx.loc[base, 'n_objects']})")

## Summary

The classical route and the U-Net agree closely on pixels, and both are limited by the same thing: touching nuclei in the clustered regime. Pixel metrics do not show this, which is why the count error is measured alongside Dice.

The two description routes differ in kind rather than in quality. The vision model produces fluent text that cannot be checked. The numbers-first route produces text where every claim traces back to a measured value, and a second model can audit it automatically. For a biomedical workflow the second is the one that can be defended.